# 5.2 图编译 Debug 信息保存功能

## 1. 功能简介

图编译问题通常需要同时观察 Dynamo、FX Graph、TorchAir 后端和设备侧编译产物。TorchAir 提供了对 PyTorch 原生 `TORCH_COMPILE_DEBUG` 的复用：设置为 `1` 后，会自动收集必要的日志和图文件。该变量必须在导入 `torchair` 和 logger 前设置。

调试产物默认位于当前目录的 `torch_compile_debug/run_<时间>-pid_<进程号>` 下。不同模式和版本生成的文件略有差异，常见内容包括 AOT 前后 GraphModule、各 Pass 的 FX 图、GE 图和编译日志。

![图编译示意图](./images/Fgraph_compile_1.png)

开启本功能后，图编译过程中能自动收集的关键调试信息如上图所示 ，详细说明参见下表。

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">信息类型</th>
    <th align="left">说明</th>
  </tr>
  <tr>
    <td align="left">日志信息</td>
    <td align="left">PyTorch原生Dynamo日志<br>npugraph_ex 日志</td>
  </tr>
  <tr>
    <td align="left">Debug信息</td>
    <td align="left">AOT前的GraphModule<br>AOT后的GraphModule<br>每个公共Pass处理后的FX图（txt文件）<br>aclgraph优化中不同Pass处理后的FX图（txt文件）<br>aclgraph编译后的FX图结构信息（output_code.py文件）<br>aclgraph在Capture阶段捕获的模型运行实例信息（原始*.json文件及用于可视化的*.aligned.json文件）<br>注意：仅当配套的CANN版本是8.5.0及之后的版本，才会有该文件生成，否则不会生成。</td>
  </tr>
</table>

## 2. 使用约束

- 使用compile_fx开启该功能时，不会收集AOT前的GraphModule文件dynamo_out_graph.txt。
- 分布式场景下，需在脚本开头import npugraph_ex，确保debug目录能正确创建。

## 3. 使用方法

在执行脚本前，可以先在终端设置环境变量：

```bash
export TORCH_COMPILE_DEBUG=1
python main.py
```

也可以在 Python 脚本中通过 `os.environ` 设置，但**必须在导入 torchair 和 logger 之前**完成。下面是一个完整的使用示例，主要步骤如下：

- **设置环境变量**：通过 `os.environ` 将 `TORCH_COMPILE_DEBUG` 设为 1。
- **开启日志**：将 logger 级别设为 DEBUG，并通过 `torch._logging.set_logs` 开启 Dynamo 和 AOT 日志。
- **配置编译选项**：通过 `CompilerConfig` 设置编译模式为 `max-autotune`，并开启 `graph_dump` 保存 GE 图。
- **编译并运行模型**：使用 `torchair.get_npu_backend` 获取后端，编译一个矩阵乘加 ReLU 的简单模型，以 `fullgraph=True`、`dynamic=False` 执行一次推理。
- **归档产物**：推理完成后，检查当前目录下自动生成的 `torch_compile_debug/run_*` 目录，将日志、FX/GE 图和版本信息一起归档。

> 运行下方代码单元格，观察输出并检查产物目录。

In [ ]:
import os
os.environ["TORCH_COMPILE_DEBUG"] = "1"

import logging
import torch
import torch_npu
import torchair
from torchair import logger

logger.setLevel(logging.DEBUG)
torch._logging.set_logs(dynamo=logging.DEBUG, aot=logging.DEBUG, output_code=True, graph_code=True)

config = torchair.CompilerConfig()
config.mode.value = "max-autotune"
config.debug.graph_dump.type = "pbtxt"
npu_backend = torchair.get_npu_backend(compiler_config=config)

class Demo(torch.nn.Module):
    def forward(self, x):
        return torch.relu(x @ x.transpose(-1, -2))

assert torch.npu.is_available(), "请在昇腾 NPU 环境运行本示例"
model = Demo().npu().eval()
compiled = torch.compile(model, backend=npu_backend, fullgraph=True, dynamic=False)
x = torch.randn(8, 32, dtype=torch.float16, device="npu:0")
with torch.no_grad():
    out = compiled(x)
torch.npu.synchronize()
print("output shape:", tuple(out.shape))
print("请检查当前目录的 torch_compile_debug/run_*/torchair 目录，并将日志、FX/GE 图和版本信息一起归档。")

## 4. 产物目录结构说明

运行示例脚本后，编译过程中必要的 Debug 信息产物目录结构如下，仅供参考，具体取决于实际开启的 Pass 数量。`torch_compile_debug` 为 PyTorch 原生开启环境变量时创建的目录，默认在当前脚本路径下。分布式场景下，运行目录名会在末尾追加 `-rank_<rank_id>`（例如 `torch_compile_debug/run_<时间>-pid_<进程号>-rank_<rank_id>`）。

```txt
torch_compile_debug
├── run_<时间>-pid_<进程号>
│   ├── npugraph_ex
│   │   ├── debug.log                                                 # npugraph_ex 日志
│   │   ├── model__0                                                  # model__0 为模型 ID
│   │   │   ├── forward                                               # 前向推理
│   │   │   │   ├── output_code.py                                    # 编译后的图结构文件
│   │   │   │   ├── 000_aot_forward_graph.txt                         # AOT 后的 GraphModule
│   │   │   │   ├── 001_aot_forward_graph_after_${pass1_name}.txt     # 公共图优化过程中每个 Pass 的输出 FX 图
│   │   │   │   ├── 002_aot_forward_graph_after_${pass2_name}.txt
│   │   │   │   ├── 003_aot_forward_graph_after_${pass5_name}.txt     # aclgraph 优化中不同 Pass 处理后的 FX 图
│   │   │   │   ├── 004_aot_forward_graph_after_${pass6_name}.txt
│   │   │   │   ├── ......                                            # 其他 Pass 优化
│   │   │   ├── dynamo_out_graph.txt                                  # AOT 前的 GraphModule
│   │   │   ├── graph_1_id_${aclgraph_id}_rank_${rank_id}_pid_${pid}_ts_${timestamp}.json              # 捕获的模型运行实例原始 JSON
│   │   │   ├── graph_1_id_${aclgraph_id}_rank_${rank_id}_pid_${pid}_ts_${timestamp}.aligned.json     # 对齐后的 JSON，用于可视化
│   │   └── torchdynamo
│       └── debug.log                                                 # Torch 原生 Dynamo 日志
└── ...                                                               # Torch 原生其他产物
```

> 模型运行实例信息通过 Runtime `aclmdlRIDebugJsonPrint` 接口获取，使用方法和说明详见 CANN 社区 [CANN Runtime 运行时 API](https://hiascend.com/document/redirect/CannCommunityRuntimeApi)。

---


## 5. 课后练习

### 一、单选题

1. 开启图编译 Debug 信息保存功能，需要设置哪个环境变量？
- A. TORCHAIR_DEBUG
- B. TORCH_COMPILE_DEBUG
- C. ASCEND_GLOBAL_LOG_LEVEL
- D. TORCH_COMPILE_CACHE

2. 在 Python 脚本中通过 `os.environ` 设置该环境变量时，正确的做法是？
- A. 在模型推理完成后设置
- B. 在调用 torch.compile 时设置
- C. 必须在导入 torchair 和 logger 之前设置
- D. 任意位置设置效果相同

3. 使用 compile_fx 开启该功能时，下列哪个文件不会被收集？
- A. output_code.py
- B. 000_aot_forward_graph.txt
- C. debug.log
- D. dynamo_out_graph.txt

4. aclgraph 在 Capture 阶段捕获的模型运行实例信息（*.json 文件）的生成条件是？
- A. 配套的 CANN 版本为 8.5.0 及之后
- B. 任意 CANN 版本均会生成
- C. 仅在分布式场景下生成
- D. 需要用户手动调用 aclmdlRIDebugJsonPrint 接口

5. 分布式场景下，Debug 产物运行目录名的特点是？
- A. 与单机场景完全相同
- B. 在末尾追加 `-rank_<rank_id>`
- C. 在开头追加 rank 信息
- D. 不再生成 Debug 产物

6. 示例中 `config.debug.graph_dump.type = "pbtxt"` 的作用是？
- A. 开启 Dynamo 日志
- B. 设置编译模式为 max-autotune
- C. 保存 GE 图
- D. 导出 ONNX 模型

### 二、多选题

7. 开启本功能后，产物中可以包含哪些图文件？
- A. AOT 前的 GraphModule（dynamo_out_graph.txt）
- B. 每个公共 Pass 处理后的 FX 图
- C. aclgraph 编译后的图结构文件（output_code.py）
- D. 训练数据集样本

8. 下列关于产物目录结构的说法，正确的有哪些？
- A. npugraph_ex 的日志文件为 debug.log
- B. output_code.py 保存编译后的图结构
- C. 000_aot_forward_graph.txt 是 AOT 前的 GraphModule
- D. 分布式场景下运行目录名会追加 `-rank_<rank_id>`

9. 开启本功能后，能自动收集的调试信息包括哪些？
- A. 日志信息（Dynamo 日志、npugraph_ex 日志）
- B. AOT 后的 GraphModule
- C. aclgraph 优化中不同 Pass 处理后的 FX 图
- D. aclgraph Capture 阶段捕获的模型运行实例 JSON

10. 使用本功能进行图编译问题定位时，合理的做法有哪些？
- A. 同时观察 Dynamo、FX Graph、TorchAir 后端和设备侧编译产物
- B. 对比各 Pass 处理后的 FX 图，锁定引入问题的环节
- C. 利用 *.aligned.json 文件进行可视化分析
- D. 依赖本功能自动修复图编译错误

**运行以下代码单元查看参考答案与解析。**

In [ ]:
from pathlib import Path
answer = Path('answer/05.02_answer.txt')
print(answer.read_text(encoding='utf-8'))